In [7]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

In [8]:
from drGAT.drGAT.metrics import get_parsed_df

In [9]:
def extract_method_data_cell_or_drug(filename):
    # ファイル名から method, data, cell_or_drug を抽出する関数を想定（例: method_data_cell.sqlite3）
    parts = filename.replace(".sqlite3", "").split("_")
    method = parts[0]
    data = parts[1]
    cell_or_drug = "_".join(parts[2:])  # 複数アンダースコアを許容
    return method, data, cell_or_drug

In [10]:
ls /data/LunaLab/drGAT/Test2_leave_X_out/

GAT_ctrp_cell.sqlite3     jobs/
GAT_ctrp_drug.sqlite3     Logs/
GAT_gdsc1_cell.sqlite3    nci_cell.ipynb
GAT_gdsc1_drug.sqlite3    nohup.out
GAT_gdsc2_cell.sqlite3    run_drGAT.py
GAT_gdsc2_drug.sqlite3    submit_all.sh*
GAT_nci_cell.sqlite3      submit_slow_2.sh*
GAT_nci_drug.sqlite3      submit_slow_ones.sh*
GATv2_ctrp_cell.sqlite3   test_file.txt
GATv2_ctrp_drug.sqlite3   Transformer_ctrp_cell.sqlite3
GATv2_gdsc1_cell.sqlite3  Transformer_ctrp_drug.sqlite3
GATv2_gdsc1_drug.sqlite3  Transformer_gdsc1_cell.sqlite3
GATv2_gdsc2_cell.sqlite3  Transformer_gdsc1_drug.sqlite3
GATv2_gdsc2_drug.sqlite3  Transformer_gdsc2_cell.sqlite3
GATv2_nci_cell.sqlite3    Transformer_gdsc2_drug.sqlite3
GATv2_nci_drug.sqlite3    Transformer_nci_cell.sqlite3
generate_shs.py           Transformer_nci_drug.sqlite3


In [11]:
all_dfs = []

for file in glob.glob("/data/LunaLab/drGAT/Test2_leave_X_out/*.sqlite3"):
    method, data, cell_or_drug = extract_method_data_cell_or_drug(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )
        if df_valid.shape[0] == 0:
            # ❗ 評価指標なしのときのダミー行
            empty_row = pd.DataFrame([{
                "n_complete": 0,
                "method": method,
                "data": data,
                "cell_or_drug": cell_or_drug,
            }])
            all_dfs.append(empty_row)
            continue

        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        df_params = df_valid[[c for c in df_valid.columns if "params" in c]].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        parsed_df["method"] = method
        parsed_df["data"] = data
        parsed_df["cell_or_drug"] = cell_or_drug

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 試行ありの部分だけで idxmax をとる
    valid_summary_df = summary_df[summary_df["n_complete"] > 0].copy()
    best_idxs = valid_summary_df.groupby(["data", "cell_or_drug", "method"])["AUPR_num"].idxmax()
    best_valid_df = summary_df.loc[best_idxs]

    # 試行ゼロの行だけ抽出
    zero_trial_df = summary_df[summary_df["n_complete"] == 0]

    # 両者を結合
    best_df = pd.concat([best_valid_df, zero_trial_df], ignore_index=True).drop(columns=["AUPR_num"])

    # インデックスと並び順
    best_df.set_index(["data", "cell_or_drug", "method"], inplace=True)
    best_df.sort_index(level=["data", "cell_or_drug"], inplace=True)

    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    desired_data_order = ["nci", "gdsc1", "gdsc2", "ctrp"]
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: (desired_data_order.index(x[0]), x[1]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_ctrp_cell.sqlite3: 3 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_ctrp_drug.sqlite3: 6 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_gdsc1_cell.sqlite3: 2 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_gdsc1_drug.sqlite3: 7 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_gdsc2_cell.sqlite3: 5 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_gdsc2_drug.sqlite3: 29 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_nci_cell.sqlite3: 105 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GAT_nci_drug.sqlite3: 11 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GATv2_ctrp_cell.sqlite3: 0 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GATv2_ctrp_drug.sqlite3: 2 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GATv2_gdsc1_cell.sqlite3: 3 trials completed
✅ /data/LunaLab/drGAT/Test2_leave_X_out/GATv2_gdsc1_drug.sqlite3: 8 trials completed


n_complete              ACC        Precision  \
data  cell_or_drug method                                                      
nci   cell         GAT                 105  0.503 (± 0.347)      0.0 (± 0.0)   
                   GATv2               106  0.503 (± 0.347)  0.018 (± 0.132)   
                   Transformer          97  0.513 (± 0.319)  0.558 (± 0.386)   
      drug         GAT                  11  0.846 (± 0.135)  0.865 (± 0.157)   
                   GATv2                 8  0.851 (± 0.138)  0.888 (± 0.152)   
                   Transformer          10  0.862 (± 0.129)   0.86 (± 0.152)   
gdsc1 cell         GAT                   2  0.561 (± 0.257)      0.0 (± 0.0)   
                   GATv2                 3   0.51 (± 0.263)  0.196 (± 0.283)   
                   Transformer           0              NaN              NaN   
      drug         GAT                   7  0.678 (± 0.122)   0.612 (± 0.23)   
                   GATv2                 8  0.638 (± 0.134)  0.552 (± 0.211)   
                   Transformer           6  0.683 (± 0.121)  0.626 (± 0.234)   
gdsc2 cell         GAT                   5  0.562 (± 0.294)  0.192 (± 0.384)   
                   GATv2                 4  0.533 (± 0.313)  0.002 (± 0.045)   
                   Transformer           5   0.66 (± 0.212)  0.561 (± 0.373)   
      drug         GAT                  29  0.761 (± 0.132)  0.726 (± 0.222)   
                   GATv2                20  0.753 (± 0.133)  0.708 (± 0.223)   
                   Transformer          19  0.762 (± 0.132)   0.72 (± 0.229)   
ctrp  cell         GAT                   3  0.584 (± 0.305)    0.07 (± 0.24)   
                   GATv2                 0              NaN              NaN   
                   Transformer           2   0.55 (± 0.317)   0.001 (± 0.03)   
      drug         GAT                   6  0.777 (± 0.131)  0.763 (± 0.191)   
                   GATv2                 2  0.786 (± 0.126)  0.761 (± 0.177)   
                   Transformer           4  0.784 (± 0.126)  0.751 (± 0.176)   

                                         Recall               F1  \
data  cell_or_drug method                                          
nci   cell         GAT              0.0 (± 0.0)      0.0 (± 0.0)   
                   GATv2          0.0 (± 0.001)    0.0 (± 0.002)   
                   Transformer  0.086 (± 0.127)  0.111 (± 0.124)   
      drug         GAT          0.838 (± 0.186)  0.836 (± 0.151)   
                   GATv2        0.813 (± 0.202)  0.832 (± 0.166)   
                   Transformer  0.877 (± 0.159)  0.857 (± 0.139)   
gdsc1 cell         GAT              0.0 (± 0.0)      0.0 (± 0.0)   
                   GATv2        0.426 (± 0.491)  0.243 (± 0.325)   
                   Transformer              NaN              NaN   
      drug         GAT          0.678 (± 0.204)  0.615 (± 0.196)   
                   GATv2        0.814 (± 0.163)   0.632 (± 0.19)   
                   Transformer  0.669 (± 0.202)  0.618 (± 0.189)   
gdsc2 cell         GAT           0.06 (± 0.168)  0.079 (± 0.192)   
                   GATv2          0.0 (± 0.002)    0.0 (± 0.004)   
                   Transformer  0.532 (± 0.371)  0.476 (± 0.318)   
      drug         GAT          0.726 (± 0.232)  0.689 (± 0.199)   
                   GATv2         0.76 (± 0.213)  0.697 (± 0.189)   
                   Transformer  0.766 (± 0.205)  0.704 (± 0.193)   
ctrp  cell         GAT          0.058 (± 0.223)  0.054 (± 0.202)   
                   GATv2                    NaN              NaN   
                   Transformer    0.0 (± 0.004)    0.0 (± 0.007)   
      drug         GAT          0.796 (± 0.184)  0.762 (± 0.168)   
                   GATv2         0.826 (± 0.16)  0.777 (± 0.152)   
                   Transformer  0.845 (± 0.148)  0.781 (± 0.147)   

                                          AUROC             AUPR  \
data  cell_or_drug method                                          
nci   cell         GAT          0.527 (± 0.067

In [12]:
output_df = best_df[desired_order].reset_index()
output_df

,data,cell_or_drug,method,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR
0,nci,cell,GAT,105,0.503 (± 0.347),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.527 (± 0.067),0.526 (± 0.338)
1,nci,cell,GATv2,106,0.503 (± 0.347),0.018 (± 0.132),0.0 (± 0.001),0.0 (± 0.002),0.51 (± 0.095),0.521 (± 0.312)
2,nci,cell,Transformer,97,0.513 (± 0.319),0.558 (± 0.386),0.086 (± 0.127),0.111 (± 0.124),0.577 (± 0.158),0.585 (± 0.288)
3,nci,drug,GAT,11,0.846 (± 0.135),0.865 (± 0.157),0.838 (± 0.186),0.836 (± 0.151),0.915 (± 0.12),0.922 (± 0.113)
4,nci,drug,GATv2,8,0.851 (± 0.138),0.888 (± 0.152),0.813 (± 0.202),0.832 (± 0.166),0.917 (± 0.117),0.924 (± 0.111)
5,nci,drug,Transformer,10,0.862 (± 0.129),0.86 (± 0.152),0.877 (± 0.159),0.857 (± 0.139),0.924 (± 0.111),0.93 (± 0.105)
6,gdsc1,cell,GAT,2,0.561 (± 0.257),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.527 (± 0.14),0.515 (± 0.245)
7,gdsc1,cell,GATv2,3,0.51 (± 0.263),0.196 (± 0.283),0.426 (± 0.491),0.243 (± 0.325),0.513 (± 0.143),0.505 (± 0.251)
8,gdsc1,cell,Transformer,0,NaN,NaN,NaN,NaN,NaN,NaN
9,gdsc1,drug,GAT,7,0.678 (± 0.122),0.612 (± 0.23),0.678 (± 0.204),0.615 (± 0.196),0.755 (± 0.148),0.704 (± 0.215)


In [13]:
output_df.sort_values('cell_or_drug')

,data,cell_or_drug,method,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR
0,nci,cell,GAT,105,0.503 (± 0.347),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.527 (± 0.067),0.526 (± 0.338)
1,nci,cell,GATv2,106,0.503 (± 0.347),0.018 (± 0.132),0.0 (± 0.001),0.0 (± 0.002),0.51 (± 0.095),0.521 (± 0.312)
2,nci,cell,Transformer,97,0.513 (± 0.319),0.558 (± 0.386),0.086 (± 0.127),0.111 (± 0.124),0.577 (± 0.158),0.585 (± 0.288)
20,ctrp,cell,Transformer,2,0.55 (± 0.317),0.001 (± 0.03),0.0 (± 0.004),0.0 (± 0.007),0.582 (± 0.166),0.556 (± 0.27)
19,ctrp,cell,GATv2,0,NaN,NaN,NaN,NaN,NaN,NaN
6,gdsc1,cell,GAT,2,0.561 (± 0.257),0.0 (± 0.0),0.0 (± 0.0),0.0 (± 0.0),0.527 (± 0.14),0.515 (± 0.245)
7,gdsc1,cell,GATv2,3,0.51 (± 0.263),0.196 (± 0.283),0.426 (± 0.491),0.243 (± 0.325),0.513 (± 0.143),0.505 (± 0.251)
8,gdsc1,cell,Transformer,0,NaN,NaN,NaN,NaN,NaN,NaN
18,ctrp,cell,GAT,3,0.584 (± 0.305),0.07 (± 0.24),0.058 (± 0.223),0.054 (± 0.202),0.533 (± 0.129),0.511 (± 0.303)
12,gdsc2,cell,GAT,5,0.562 (± 0.294),0.192 (± 0.384),0.06 (± 0.168),0.079 (± 0.192),0.69 (± 0.199),0.691 (± 0.271)
